# New-corpus gold validator

**Run this before spending on retrieval.** For a `gold_sections`-style question set,
it checks that every gold section resolves to a node in your index — catching the
case where an LLM-inferred tree dropped its section numbers (e.g. the vanilla PDF arm:
point `INDEX` at `IDX-PDF-vanilla-rfc9110` to see 68/68 UNMAPPABLE).

**Kernel:** `pandas` + `altair` (`.venv`). Reuses `score_recall`'s tested gold parser
and addressing detector — read-only.

In [ ]:
from pathlib import Path
import sys, json, csv

def find_repo(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "indexes").exists() and (p / "scripts" / "score_recall.py").exists():
            return p
    raise RuntimeError("run this notebook from inside the repo")

REPO = find_repo()
sys.path.insert(0, str(REPO / "scripts"))
import pandas as pd
import altair as alt
import viz_theme
import score_recall as SR           # reuse the tested gold parser + addressing detector
viz_theme.register()
P = viz_theme.PALETTE

# ---- point these at the index + gold set you want to validate ----
INDEX     = "IDX-PDF-outline-rfc9110"                 # any indexes/<id>/
QUESTIONS = REPO / "evaluations" / "questions-rfc9110.csv"

struct = json.load((REPO / "indexes" / INDEX / "index.json").open())["structure"]
ADDR = SR.detect_addressing(struct)                   # "line" or "node" (page)

def walk(nodes):                                      # trivial tree walk (not metric logic)
    for n in nodes:
        yield n
        yield from walk(n.get("nodes", []) or [])

# section number -> node, by matching the leading dotted id in each node title
sec2node = {}
for n in walk(struct):
    m = SR.SEC_RE.match(n.get("title", ""))
    if m:
        sec2node.setdefault(m.group(1), n)

fields = next(csv.reader(QUESTIONS.open()))
SCHEMA = "structural" if "gold_sections" in fields else "prose"
print(f"index={INDEX} addressing={ADDR} indexed_sections={len(sec2node)}")
print(f"gold schema = {SCHEMA}  ({QUESTIONS.name})")

## Resolve each gold section to a node

One row per (question, gold section): does it
map to a node in this index, and where? `UNMAPPABLE` = the retriever literally cannot
reach that section here.

In [ ]:
assert SCHEMA == "structural", "this cell is for gold_sections corpora (RFC-style)"

def locator(node):
    return (str(node.get("line_num")) if ADDR == "line"
            else f"p{node.get('start_index')}-{node.get('end_index')}")

rows = []
for q in csv.DictReader(QUESTIONS.open()):
    gold, is_absence = SR.gold_list(q.get("gold_sections", ""))
    for sec in gold:
        node = sec2node.get(sec)
        rows.append(dict(qid=q["id"], category=q.get("category", ""), section=sec,
                         resolved=node is not None,
                         node_title=(node or {}).get("title", "")[:48],
                         locator=locator(node) if node else "—"))
res = pd.DataFrame(rows)
n_ok, n_tot = int(res.resolved.sum()), len(res)
print(f"{n_ok}/{n_tot} gold sections resolve to a node in {INDEX}."
      + ("  ✓ safe to run retrieval." if n_ok == n_tot
         else f"  ✗ {n_tot - n_ok} UNMAPPABLE — this index lost those labels; retrieval can't reach them."))
res.head(12)

## Coverage

In [ ]:
# coverage as a single-hue magnitude bar (status on the axis, count = length)
cov = (res.assign(status=res.resolved.map({True: "resolved", False: "UNMAPPABLE"}))
          .groupby("status", as_index=False).size())
bars = alt.Chart(cov).mark_bar().encode(
    x=alt.X("size:Q", title="gold sections"),
    y=alt.Y("status:N", sort="-x", title=None),
    tooltip=["status", "size"])
labels = bars.mark_text(align="left", dx=4, color=P["ink"]).encode(text="size:Q")
(bars + labels).properties(title=f"Gold coverage on {INDEX}", width=420, height=110)

## UNMAPPABLE sections — the blockers to fix

In [ ]:
# the actionable output: exactly which (question, section) pairs won't resolve.
unmapped = res[~res.resolved][["qid", "category", "section"]]
if unmapped.empty:
    print("No UNMAPPABLE gold sections — every gold section maps to a node. Good to go.")
else:
    print(f"{len(unmapped)} UNMAPPABLE gold section(s) — fix the index/labels before spending:")
unmapped

## Spot-check a mapping

Resolution is by title match; confirm the mapped node holds the
expected text (guards against a wrong-but-numbered heading).

In [ ]:
# spot-check a resolved section: does the mapped node actually hold the right text?
sec = res[res.resolved].iloc[0]["section"]
node = sec2node[sec]
print(f"section {sec}  ->  {node['title']!r}  [{('line '+str(node.get('line_num'))) if ADDR=='line' else 'pp '+str(node.get('start_index'))+'-'+str(node.get('end_index'))}]")
print("\n" + " ".join((node.get("text") or "(no inline text)").split())[:400])

## Prose-gold corpora (site / paper)

Those sets use free-text evidence, not section ids —
not structurally checkable. This cell does presence checks and says so plainly.

In [ ]:
# Prose-gold corpora (site / paper) use free-text expected_evidence + ground_truth,
# which CANNOT be structurally validated against the tree. Only presence/format checks.
if SCHEMA == "prose":
    q = pd.DataFrame(csv.DictReader(QUESTIONS.open()))
    checks = pd.DataFrame({
        "has_expected_evidence": q.get("expected_evidence", "").astype(bool) if "expected_evidence" in q else False,
        "has_ground_truth": q.get("ground_truth", "").astype(bool) if "ground_truth" in q else False,
    })
    print(f"{len(q)} questions; prose gold — no structural check possible. Presence:")
    display(checks.sum().to_frame("n_present"))
else:
    print("Structural (gold_sections) corpus — see the cells above; nothing prose to check.")